# ME6030 — Markov Matrix Analysis
### Manufacturing Plant Workstation Transition Problem
**Shailesh K | me22b192**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
os.makedirs('plots', exist_ok=True)

## 1. Define the Markov Transition Matrix

In [ ]:
# P[i,j] = probability of transitioning FROM state j TO state i
P = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.5, 0.2],
    [0.0, 0.3, 0.7]
])

print("Transition matrix P:")
print(P)
print("\nColumn sums (must all equal 1 for a Markov matrix):")
print(P.sum(axis=0))

## 2. Eigenvalues and Eigenvectors

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(P)

# Sort by descending magnitude so lambda=1 appears first
idx = np.argsort(np.abs(eigenvalues))[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues (sorted by descending magnitude):")
for i, lam in enumerate(eigenvalues):
    print(f"  lambda_{i+1} = {lam.real:.6f}  (|lambda| = {np.abs(lam):.6f})")

print("\nEigenvectors (columns correspond to eigenvalues above):")
print(np.round(eigenvectors.real, 6))

## 3. Steady-State Eigenvector (lambda = 1)

In [ ]:
# Extract eigenvector for lambda=1 (first column after sorting)
ss_vector = eigenvectors[:, 0].real

# Ensure positive entries (eigenvectors are defined up to sign)
if ss_vector[0] < 0:
    ss_vector = -ss_vector

# L1-normalise so entries sum to 1 (probability interpretation)
ss_vector = ss_vector / ss_vector.sum()

labels = ['State 1 (Machining)', 'State 2 (Inspection)', 'State 3 (Rework)']

print("Steady-state distribution pi (eigenvector for lambda=1, normalised):")
for label, val in zip(labels, ss_vector):
    print(f"  {label}: {val:.6f}  ({val*100:.2f}%)")

print(f"\nSum check: {ss_vector.sum():.10f}")

## 4. Magnitudes of All Three Eigenvalues

In [ ]:
print("Eigenvalue magnitudes:")
for i, lam in enumerate(eigenvalues):
    print(f"  |lambda_{i+1}| = |{lam.real:.6f}| = {np.abs(lam):.6f}")

print("\nSince |lambda_1| = 1 and |lambda_2|, |lambda_3| < 1,")
print("the transient modes decay geometrically and the system converges")
print("to a unique steady-state distribution.")

## 5. Plot: Eigenvalue Magnitudes

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

mags = np.abs(eigenvalues)
mag_labels = [f'$\\lambda_{i+1}$ = {eigenvalues[i].real:.4f}' for i in range(3)]
colors = ['#2ecc71' if abs(m - 1.0) < 1e-6 else '#3498db' for m in mags]

bars = ax.bar(mag_labels, mags, color=colors, edgecolor='black', linewidth=0.8)
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.2, label='$|\\lambda| = 1$ boundary')
ax.set_ylabel('$|\\lambda|$')
ax.set_title('Eigenvalue Magnitudes of Markov Matrix $P$')
ax.set_ylim(0, 1.25)
ax.legend()
ax.grid(axis='y', alpha=0.4)

for bar, val in zip(bars, mags):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('plots/eigenvalue_magnitudes.png', bbox_inches='tight')
plt.show()
print("Saved: plots/eigenvalue_magnitudes.png")

## 6. Plot: Steady-State Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

short_labels = ['State 1\n(Machining)', 'State 2\n(Inspection)', 'State 3\n(Rework)']
colors_ss = ['#e74c3c', '#3498db', '#f39c12']
bars = ax.bar(short_labels, ss_vector, color=colors_ss, edgecolor='black', linewidth=0.8)

ax.set_ylabel('Probability')
ax.set_title('Steady-State Distribution of Workstation Occupancy')
ax.set_ylim(0, max(ss_vector) * 1.3)
ax.grid(axis='y', alpha=0.4)

for bar, val in zip(bars, ss_vector):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}\n({val*100:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plots/steady_state_distribution.png', bbox_inches='tight')
plt.show()
print("Saved: plots/steady_state_distribution.png")

## 7. Simulate the Difference Equation: z_k = P^k * z_0

In [ ]:
# Initial condition: job enters at machining centre (State 1)
z0 = np.array([1.0, 0.0, 0.0])
steps = 10000

# Non-uniform recording: dense at start, sparse later (log-scale plot)
record_set = set(
    list(range(0, 51)) +
    list(range(51, 201, 5)) +
    list(range(201, 1001, 10)) +
    list(range(1001, steps + 1, 100))
)
record_steps = sorted(record_set)

z_history = np.zeros((len(record_steps), 3))
z = z0.copy()
rec_idx = 0

for k in range(steps + 1):
    if k in record_set:
        z_history[rec_idx] = z
        rec_idx += 1
    if k < steps:
        z = P @ z

print(f"Initial condition z_0 = {z0}")
print(f"z_10000 = {np.round(z, 8)}")
print(f"Steady-state pi = {np.round(ss_vector, 8)}")
print(f"\n|z_10000 - pi| = {np.abs(z - ss_vector)}")
print(f"Converged (tol=1e-6): {np.allclose(z, ss_vector, atol=1e-6)}")

## 8. Plot: Convergence of All Three States

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

state_colors = ['#e74c3c', '#3498db', '#f39c12']
state_plot_labels = ['State 1 (Machining)', 'State 2 (Inspection)', 'State 3 (Rework)']

for i in range(3):
    ax.plot(record_steps, z_history[:, i],
            color=state_colors[i], linewidth=1.8, label=state_plot_labels[i])
    ax.axhline(y=ss_vector[i], color=state_colors[i],
               linestyle='--', linewidth=0.9, alpha=0.7)

ax.set_xlabel('Step $k$')
ax.set_ylabel('$z_k$ (state probability)')
ax.set_title('Convergence of State Distribution: $z_k = P^k z_0$')
ax.legend(loc='upper right')
ax.set_xscale('log')
ax.set_xlim(left=1)
ax.grid(alpha=0.3)
ax.text(0.02, 0.02, 'Dashed lines = steady-state values $\\pi$',
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('plots/convergence_all_states.png', bbox_inches='tight')
plt.show()
print("Saved: plots/convergence_all_states.png")

## 9. Plot: All Three Eigenvectors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
comp_labels = ['S1\n(Mach.)', 'S2\n(Insp.)', 'S3\n(Rework)']

for i, ax in enumerate(axes):
    vec = eigenvectors[:, i].real
    clrs = ['#2ecc71' if v >= 0 else '#e74c3c' for v in vec]
    ax.bar(comp_labels, vec, color=clrs, edgecolor='black', linewidth=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Eigenvector {i+1}\n($\\lambda_{i+1}$ = {eigenvalues[i].real:.4f})')
    if i == 0:
        ax.set_ylabel('Component value')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Eigenvectors of Markov Matrix $P$', fontsize=12)
plt.tight_layout()
plt.savefig('plots/eigenvectors_bar.png', bbox_inches='tight')
plt.show()
print("Saved: plots/eigenvectors_bar.png")

## 10. Summary

In [ ]:
print("=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"\nEigenvalues:  {np.round(eigenvalues.real, 6)}")
print(f"Magnitudes:   {np.round(np.abs(eigenvalues), 6)}")
print(f"\nSteady-state distribution pi:")
for label, val in zip(labels, ss_vector):
    print(f"  {label}: {val:.6f}")
print(f"\nz_10000 matches pi: {np.allclose(z, ss_vector, atol=1e-6)}")
print(f"\nAll plots saved to: plots/")